# Starting to analyze the data of 2009 


The core of this algo is to loop over every dimension possible and inside loop over all the different type of functions,  print the name of the function, its associated ERT at the time where it reaches the smallest target precision.  



In [3]:
import cocopp
dsl = cocopp.load("bbob/2009/*")

# 1) Determine what is the best algo for each dimension, function, and for a target precision of 1e-08.

This code will be looping over each dimension and each type of function. 
The goal here is mainly to get a general understnad of the different types of functions that can be used to call the data, but also how the data is presented/ how it looks like. 

In [4]:
import numpy as np

dd = dsl.dictByDimFunc()     # your grouped datasets
t = 1e-8                     # choose the target precision

best_by_df = {}              # (dim, fid) -> (best_alg, best_ert)

for dim in sorted(dd.keys()): 
    for fid in sorted(dd[dim].keys()):
        rows = []
        for ds in dd[dim][fid]:                 # each ds = one algorithm
            ert = float(ds.detERT([t])[0])      # ERT in #evals at target t
            rows.append((ds.algId, ert))  
        # ignore INF (not reached) when picking best
        finite = [(a, e) for (a, e) in rows if np.isfinite(e)] 
    
        if finite:
            best_alg, best_ert = min(finite, key=lambda x: x[1]) 
        else:
            best_alg, best_ert = None, np.inf
        best_by_df[(dim, fid)] = (best_alg, best_ert) 
        print(f"dim={dim:>2}, F{fid:>2} -> {best_alg}  (ERT={best_ert:.3g} @ {t})")


dim= 2, F 1 -> NEWUOA_ros  (ERT=6.2 @ 1e-08)
dim= 2, F 2 -> LSfminbnd_posik  (ERT=30 @ 1e-08)
dim= 2, F 3 -> LSstep_posik  (ERT=465 @ 1e-08)
dim= 2, F 4 -> LSstep_posik  (ERT=569 @ 1e-08)
dim= 2, F 5 -> MCS_huyer  (ERT=4.4 @ 1e-08)
dim= 2, F 6 -> NELDERDOERR_doerr  (ERT=136 @ 1e-08)
dim= 2, F 7 -> CMA-ESPLUSSEL_auger  (ERT=258 @ 1e-08)
dim= 2, F 8 -> MCS_huyer  (ERT=116 @ 1e-08)
dim= 2, F 9 -> MCS_huyer  (ERT=94.2 @ 1e-08)
dim= 2, F10 -> NELDERDOERR_doerr  (ERT=104 @ 1e-08)
dim= 2, F11 -> NELDERDOERR_doerr  (ERT=105 @ 1e-08)
dim= 2, F12 -> NELDER_hansen  (ERT=211 @ 1e-08)
dim= 2, F13 -> NELDER_hansen  (ERT=133 @ 1e-08)
dim= 2, F14 -> NELDERDOERR_doerr  (ERT=101 @ 1e-08)
dim= 2, F15 -> NELDERDOERR_doerr  (ERT=1.46e+03 @ 1e-08)
dim= 2, F16 -> NELDERDOERR_doerr  (ERT=560 @ 1e-08)
dim= 2, F17 -> BIPOP-CMA-ES_hansen  (ERT=1.83e+03 @ 1e-08)
dim= 2, F18 -> BIPOP-CMA-ES_hansen  (ERT=2.93e+03 @ 1e-08)
dim= 2, F19 -> MCS_huyer  (ERT=282 @ 1e-08)
dim= 2, F20 -> MCS_huyer  (ERT=376 @ 1e-08)
dim= 2

# 2) Building a counter to count the number of times an algo appears as best 

In [5]:
from collections import Counter, defaultdict

In [6]:
# Build a frequency counter: how many (dim,fid) each algo wins
win_counter = Counter(
    alg for (alg, ert) in best_by_df.values()
    if alg is not None and np.isfinite(ert)
)

# If you want a plain dict:
wins_dict = dict(win_counter)

# (Optional) pretty print, most wins first
for alg, count in win_counter.most_common():
    print(f"{alg}: {count}")

BIPOP-CMA-ES_hansen: 23
NEWUOA_ros: 16
IPOP-SEP-CMA-ES_ros: 14
MCS_huyer: 13
NELDERDOERR_doerr: 13
LSstep_posik: 10
iAMALGAM_bosman: 10
FULLNEWUOA_ros: 8
NELDER_hansen: 7
LSfminbnd_posik: 6
CMA-ESPLUSSEL_auger: 5
GLOBAL_pal: 5
AMALGAM_bosman: 5
BFGS_ros: 3
DASA_korosec: 2
MA-LS-CHAIN_molina: 2
ONEFIFTH_auger: 1


# 3) Determine what is the best algo depending on the dimension. 

For each dimension, We will determine which is the best algo ( still working with a target precision of e 10-08 ). We will go through all the different functions and decide which one is the best (in the sense appears the most).

In [7]:
"""
Given best_by_df: {(dim, fid): (alg, ert)},
return {dim: algo_with_most_(fid)_wins_in_that_dim}.
Tie-break: lower total ERT across that dim, then alphabetical.
    """
wins = defaultdict(Counter)                    # dim -> Counter({alg: count})
ert_sums = defaultdict(lambda: defaultdict(float))  # dim -> {alg: total_ert}

for (dim, fid), (alg, ert) in best_by_df.items():
    if alg is None or not np.isfinite(ert):
        continue
    wins[dim][alg] += 1
    ert_sums[dim][alg] += float(ert)

result = {}
for dim, counter in wins.items():
    max_wins = max(counter.values())
    candidates = [a for a, c in counter.items() if c == max_wins]
    best = min(candidates, key=lambda a: (ert_sums[dim][a], a))  # tie-breaks
    result[dim] = best
result


{2: 'NELDERDOERR_doerr',
 3: 'NELDERDOERR_doerr',
 5: 'IPOP-SEP-CMA-ES_ros',
 10: 'BIPOP-CMA-ES_hansen',
 20: 'BIPOP-CMA-ES_hansen',
 40: 'NEWUOA_ros'}

In [8]:
import numpy as np
import pandas as pd

# Make sure 'dd' already exists
# (if not, run: dsl = cocopp.load('path/to/your/ppdata'); dd = dsl.dictByDimFunc())

targets = [1e-1, 1e-2, 1e-3, 1e-5, 1e-8]
rows = []  # reset before starting the full loop

for dim in sorted(dd.keys()):                      # e.g. [2, 3, 5, 10, 20, 40]
    for fid in sorted(dd[dim].keys()):
        for t in targets:
            algo_erts = []
            for ds in dd[dim][fid]:                # each algorithm
                ert = float(ds.detERT([t])[0])
                algo_erts.append((ds.algId, ert))
            
            finite = [(a, e) for (a, e) in algo_erts if np.isfinite(e)]

            if finite:
                best_alg, best_ert = min(finite, key=lambda x: x[1])
            else:
                best_alg, best_ert = None, np.inf

            rows.append({
                "dimension": dim,
                "function_id": fid,
                "target": t,
                "best_algorithm": best_alg,
                "best_ERT": best_ert
            })

# Build DataFrame
df_best = pd.DataFrame(rows)
df_best = df_best.sort_values(by=["dimension", "function_id", "target"]).reset_index(drop=True)

# Confirm dimensions included
print(" Unique dimensions in table:", df_best["dimension"].unique())
print(df_best.head(15))


 Unique dimensions in table: [ 2  3  5 10 20 40]
    dimension  function_id        target   best_algorithm    best_ERT
0           2            1  1.000000e-08       NEWUOA_ros    6.200000
1           2            1  1.000000e-05       NEWUOA_ros    6.200000
2           2            1  1.000000e-03       NEWUOA_ros    6.200000
3           2            1  1.000000e-02       NEWUOA_ros    6.200000
4           2            1  1.000000e-01       NEWUOA_ros    5.666667
5           2            2  1.000000e-08  LSfminbnd_posik   30.000000
6           2            2  1.000000e-05  LSfminbnd_posik   28.000000
7           2            2  1.000000e-03  LSfminbnd_posik   25.800000
8           2            2  1.000000e-02  LSfminbnd_posik   25.333333
9           2            2  1.000000e-01  LSfminbnd_posik   24.666667
10          2            3  1.000000e-08     LSstep_posik  465.466667
11          2            3  1.000000e-05     LSstep_posik  454.066667
12          2            3  1.000000e-03 

#### Attention there are only the 15 first lines of the table that appear ! The table is very large !

This is why I am making a file where all the data gets exported to.

In [15]:
import os
os.makedirs("results", exist_ok=True)

df_best.to_csv("results/best_algos_2009.csv", index=False)
